# Effect of thickness on flux and apparent permeability

Temperature and pressure are held at `CONDITIONS['T_operating']` and `CONDITIONS['P_upstream']` (from `study_config.py`). Three sweeps, one row each:

1. **Oxide thickness** over `L_oxide_range`, metal fixed at `L_metal`
2. **Metal thickness** over `L_metal_range`, oxide fixed at `L_oxide`
3. **Whole wall scaled**: both layers multiplied by the same factor $s$ (`L_metal_range / L_metal`), so the oxide:metal ratio stays fixed

The flux follows $J \propto L^{\,m}$. $m = -1$ when the swept thickness controls the flux, $m \approx 0$ when something else does (the other layer, or the surface).

Row 3 separates **bulk from surface**: every diffusion step scales as $1/L$, so a bulk-controlled wall gives exactly $m = -1$ and a flat $\Phi_{app}$ whichever layer dominates. Surface kinetics do not depend on thickness, so they pull $m$ toward 0 and make $\Phi_{app}$ grow with $L$.

$\Phi_{app} = J\,L_{total}/\Delta\sqrt{P}$ ($\Delta P$ for L2a) comes from the same flux. It is flat only where thickness-proportional diffusion controls, which is when the "permeability" is a material property.

## 1. Setup

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats
from calculations.config.model_config import (CONDITIONS, MICROSTRUCTURE,
                                              OXIDE_DEFECTS, build_simulation_config)
from calculations.oxide_permeation import (get_oxide_properties_at_T, get_metal_properties_at_T,
                                           molecular_diffusion_flux)
from calculations.permeation_calc import calculate_simple_metal_flux, calculate_defective_metal_flux
from calculations.interface_solver import calculate_oxide_metal_system
from calculations.parallel_oxide_defect_paths import (calculate_parallel_path_flux,
                                                      calculate_parallel_path_flux_defective_metal)
from calculations.wall_labels import label_model1, label_model2
from calculations.permeability import UNITS_SIEVERTS, UNITS_HENRY
import pandas as pd
from IPython.display import display
from calculations.figures import results_dir, save_figure
FIGS = results_dir("effect-of-thickness")                             # figures and fit tables go in this folder

SIM            = build_simulation_config()
T_op, P_up     = CONDITIONS['T_operating'], CONDITIONS['P_upstream']
P_down         = SIM['P_downstream']
L_ox0, L_m0    = SIM['L_oxide'], SIM['L_metal']       # the study's wall; each sweep moves away from it
N              = CONDITIONS['n_points']

GRID_OX = np.logspace(*np.log10(CONDITIONS['L_oxide_range']), N)            # oxide sweep (m)
GRID_M  = np.logspace(*np.log10(CONDITIONS['L_metal_range']), N)            # metal sweep (m)
SCALE   = np.logspace(*np.log10(np.array(CONDITIONS['L_metal_range']) / L_m0), N)   # whole-wall factor s

# Temperature is fixed, so the layer properties are computed once; thickness is set per solve
OX = get_oxide_properties_at_T(SIM['oxide_name'], T_op)
ME = get_metal_properties_at_T(SIM['metal_name'], T_op)

LEVELS = {   # level: (title, layers in the wall)
    'L1':  ('Level 1: Perfect metal (lattice diffusion)',          {'metal'}),
    'L2a': ('Level 2a: Perfect oxide alone (molecular H$_2$)',     {'oxide'}),
    'L2b': ('Level 2b: Perfect oxide + perfect metal',             {'oxide', 'metal'}),
    'L3':  ('Level 3: Defective oxide + perfect metal',            {'oxide', 'metal'}),
    'L4':  ('Level 4: Defective metal (traps + grain boundaries)', {'metal'}),
    'L5':  ('Level 5: Defective oxide + defective metal',          {'oxide', 'metal'}),
}

## 2. Physics (one solver call per level and thickness, then the sweeps and the fit)

In [ ]:
def solve(level, Lox, Lm):
    """Flux and regime label for one level with the given layer thicknesses (T and P fixed)."""
    if level.endswith('L6'):
        return solve_m2(level, Lox, Lm)                  # Model 2 levels, defined further down
    ox, me = dict(OX, thickness=Lox), dict(ME, thickness=Lm)
    D, K  = me['D_metal'], me['K_s_metal']
    micro = dict(microstructure_params=MICROSTRUCTURE, lattice_density=MICROSTRUCTURE['N_L'], mode='both')
    if level == 'L1':  return calculate_simple_metal_flux(D, K, Lm, P_up, P_down)['flux'], label_model1('L1')
    if level == 'L2a': return molecular_diffusion_flux(ox['D_ox'], ox['K_ox'], Lox, P_up, P_down), label_model1('L2a')
    if level == 'L2b': r = calculate_oxide_metal_system(P_up, P_down, ox, me)
    if level == 'L3':  r = calculate_parallel_path_flux(P_up, P_down, ox, me, OXIDE_DEFECTS)
    if level == 'L4':  r = calculate_defective_metal_flux(D_lattice=D, K_s=K, metal_thickness=Lm,
                                                          P_up=P_up, P_down=P_down, temperature=T_op, **micro)
    if level == 'L5':  r = calculate_parallel_path_flux_defective_metal(P_up, P_down, ox, me,
                                                                        OXIDE_DEFECTS, T_op, **micro)
    return r.get('flux_total', r.get('flux')), label_model1(level, r, P_up, P_down, ox, me, OXIDE_DEFECTS, T_op, micro)

def thicknesses(row):
    """(L_ox, L_m) arrays for one sweep; in rows 1-2 the other layer stays at the study value."""
    if row == 'oxide':
        return GRID_OX, np.full(N, L_m0)
    if row == 'metal':
        return np.full(N, L_ox0), GRID_M
    return SCALE * L_ox0, SCALE * L_m0                   # whole wall: both layers × s, ratio fixed

def sweep(level, row):
    """x (m), J, Φ_app, units, regime for one sweep, or None if the level lacks that layer."""
    layers = LEVELS[level][1]
    if row in ('oxide', 'metal') and row not in layers:
        return None
    Lox, Lm = thicknesses(row)
    J, regime = map(np.array, zip(*[solve(level, a, b) for a, b in zip(Lox, Lm)]))
    L_tot = Lox * ('oxide' in layers) + Lm * ('metal' in layers)
    x = {'oxide': Lox, 'metal': Lm, 'total': L_tot}[row]
    # Φ_app written out: apparent_permeability() takes one thickness, not an array of them
    if level == 'L2a':                                   # Henry: J·L/ΔP
        return x, J, J * L_tot / (P_up - P_down), UNITS_HENRY, regime
    return x, J, J * L_tot / (np.sqrt(P_up) - np.sqrt(P_down)), UNITS_SIEVERTS, regime

def power_law(L, y):
    """y = A·L^m, fitted as ln y = ln A + m·ln L (L in metres, so A is in SI)."""
    fit = stats.linregress(np.log(L), np.log(y))
    return fit.slope, np.exp(fit.intercept), fit.rvalue**2   # m, A, R²

## 3. Style

In [ ]:
mpl.rcParams.update({
    'font.size': 14, 'font.weight': 'bold', 'mathtext.default': 'bf',
    'axes.labelweight': 'bold', 'axes.titleweight': 'bold', 'axes.linewidth': 1.8,
    'xtick.direction': 'in', 'ytick.direction': 'in', 'ytick.right': True,
    'xtick.major.width': 1.6, 'ytick.major.width': 1.6,
    'legend.frameon': False, 'legend.fontsize': 11, 'savefig.dpi': 600, 'savefig.bbox': 'tight',
})
INK, GREY, SHADE = '#0b0b0b', '#52514e', '#e1e0d9'

# Regime labels name the dominant path and the limiting step inside it (calculations/wall_labels.py):
# colour = limiting step, hollow = a defect path carries most of the flux, diamond = trap-controlled.
from calculations.wall_labels import describe as look, fitted
UNITS_TEX = {UNITS_SIEVERTS: r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1/2}$',
             UNITS_HENRY:    r'mol m$^{-1}$ s$^{-1}$ Pa$^{-1}$'}

## 4. Plot

In [ ]:
from matplotlib.lines import Line2D

LINES, ROLES = ['-', '--', ':'], ['low', 'mid', 'high']   # fit line style per condition, in order

def p_tag(Pc):
    """Short pressure label: 1e-07 -> '1e-7 Pa', 1.0 -> '1e0 Pa', 1e+07 -> '1e7 Pa'."""
    return f'{Pc:.0e}'.replace('e+0', 'e').replace('e-0', 'e-') + ' Pa'

def framed(nrows, figsize, sharex=True):
    """Figure with two legend strips under the title and an nrows x 2 grid of panels below them."""
    fig  = plt.figure(figsize=figsize, layout='constrained')
    gs   = fig.add_gridspec(nrows + 2, 2, height_ratios=[0.001, 0.001] + [1] * nrows)
    keys = [fig.add_subplot(gs[k, :]) for k in (0, 1)]
    for k in keys:
        k.axis('off')
    ax = np.empty((nrows, 2), dtype=object)
    for r in range(nrows):
        for c in range(2):
            ax[r, c] = fig.add_subplot(gs[r + 2, c], sharex=ax[0, c] if (r and sharex) else None)
    return fig, keys, ax

def shared_legend(keys, axes, lines):
    """One legend per figure: the markers (by regime) on the first strip, the lines on the second."""
    marks = {}
    for a in axes:
        for h, label in zip(*a.get_legend_handles_labels()):
            marks.setdefault(label, h)
    keys[0].legend(list(marks.values()), list(marks), title='Simulation', loc='center', ncol=min(len(marks), 2))
    keys[1].legend(handles=lines, loc='center', ncol=len(lines))

def condition_lines(tags, fit_label):
    """Line key: one style per condition, or a single fit line when there is one condition."""
    if len(tags) == 1:
        return [Line2D([], [], color=INK, lw=2.4, label=fit_label)]
    return [Line2D([], [], color=INK, lw=2.4, ls=LINES[ci], label=f'{ROLES[ci]}: {tag}')
            for ci, tag in enumerate(tags)]

def end_label(ax, xv, y, tag, ci):
    """Name a curve at its right-hand end, staggered so coinciding curves stay readable."""
    ok = np.flatnonzero(np.isfinite(y))
    i = ok[np.argmax(xv[ok])]
    ax.annotate(tag, (xv[i], y[i]), xytext=(12, 14 * (ci - 1)), textcoords='offset points',
                va='center', fontsize=11, color=INK)

def fit_table(rows):
    """Every fitted line as one row: shown under the figure, and returned for the CSV."""
    table = pd.DataFrame(rows).replace({r'\$_2\$': '₂'}, regex=True)
    with pd.option_context('display.float_format', '{:.4g}'.format, 'display.width', 220,
                           'display.max_columns', 20):
        display(table)
    return table

EXPONENT = 'm (J) / slope (Φ_app)'

ROWS = {   # sweep: (panel title, x-axis label, metres -> axis units)
    'oxide': (f'Oxide thickness swept (metal fixed at {L_m0*1e3:g} mm)',  r'Oxide thickness $L_{ox}$ (µm)',     1e6),
    'metal': (f'Metal thickness swept (oxide fixed at {L_ox0*1e6:g} µm)', r'Metal thickness $L_m$ (mm)',        1e3),
    'total': (f'Whole wall scaled (oxide:metal fixed at 1:{L_m0/L_ox0:.0f})', r'Total thickness $L_{total}$ (mm)', 1e3),
}

def panel(ax, x, x_axis, y, regime, rows, quantity, sweep_name, tag, ls):
    """Regime-coloured simulation points with one power-law fit per regime; each fit becomes a table row."""
    for reg in dict.fromkeys(regime):
        m = regime == reg
        name, c, mk, *fill = look(reg)
        hollow = fill == ['hollow']                      # Model 2: flux mainly through oxide defects
        ax.loglog(x_axis[m], y[m], mk, ms=9, color=c, mfc='white' if hollow else c,
                  mec=c if hollow else 'white', mew=2 if hollow else 1, label=name)
        if not fitted(reg) or m.sum() < 3:
            continue                                     # drawn, not fitted
        k, A, r2 = power_law(x[m], y[m])
        ax.loglog(x_axis[m], A * x[m]**k, ls, lw=2.8, color=c)
        rows.append({'quantity': quantity, 'sweep': sweep_name, 'condition': tag, 'fit': name,
                     'L window (m)': f'{x[m].min():.2e}–{x[m].max():.2e}', 'points': int(m.sum()),
                     EXPONENT: k, 'A (SI)': A, 'R²': r2})
    ax.grid(color=SHADE)

def no_layer(ax, row):
    ax.text(0.5, 0.5, f'no {row} in this level', transform=ax.transAxes,
            ha='center', va='center', fontsize=15, color=GREY)
    ax.set_xticks([]); ax.set_yticks([])

def thickness_figure(level, curves=None, note=None):
    """3x2 figure (rows = sweeps) and its fit table. curves = {row: [(tag, sweep output or None), ...]}."""
    title = LEVELS[level][0]
    if curves is None:                                   # the study's own (T_op, P_up)
        curves = {row: [(f'{T_op:.0f} K, {p_tag(P_up)}', sweep(level, row))] for row in ROWS}
    tags = [t for t, _ in next(iter(curves.values()))]
    multi, units = len(tags) > 1, UNITS_SIEVERTS
    fig, keys, ax = framed(3, (14, 19), sharex=False)
    rows = []
    for i, (row, (row_title, x_label, to_axis)) in enumerate(ROWS.items()):
        present = [(t, out) for t, out in curves[row] if out is not None]
        for j, qty in enumerate(('J', 'Phi_app')):
            p = ax[i, j]
            p.set_title(row_title, fontsize=13)
            if not present:
                no_layer(p, row)
                continue
            units = present[0][1][3]
            for ci, (tag, (xs, J, Phi, _, regime)) in enumerate(present):
                y = J if qty == 'J' else Phi
                panel(p, xs, xs * to_axis, y, regime, rows, qty, row, tag, LINES[ci] if multi else '-')
                if multi:
                    end_label(p, xs * to_axis, y, tag, ci)
            xa = present[0][1][0] * to_axis
            p.set_xlim(xa.min() / 1.5, xa.max() * (6 if multi else 1.5))   # room for the end labels
            p.set_xlabel(x_label)
            p.set_ylabel(r'Flux $J$ (mol m$^{-2}$ s$^{-1}$)' if qty == 'J'
                         else rf'$\Phi_{{app}}$ ({UNITS_TEX[units]})')
    shared_legend(keys, ax.flat, condition_lines(tags, 'Power-law fit (colour: per regime)'))
    drive = r'\Delta P' if units == UNITS_HENRY else r'\Delta\sqrt{P}'
    where = note or rf'$T$ = {T_op:.0f} K,   $P_{{up}}$ = {P_up:.0e} Pa'
    fig.suptitle(f'{title}\n{where},   ' + rf'$J \propto L^m$,   $\Phi_{{app}} = J\,L_{{total}}/{drive}$',
                 fontsize=18, fontweight='bold')
    plt.show()
    return fig, fit_table(rows)

## 5. Model 1 (L1–L5)

In [ ]:
for level in LEVELS:
    fig, fits = thickness_figure(level)
    save_figure(fig, f"{level}_thickness", FIGS)
    fits.to_csv(FIGS / f"{level}_thickness_fits.csv", index=False)

## 6. Model 2 (L6 family)

In [ ]:
# =============================================================================
# MODEL 2 (L6 family): dissociation on the gas-facing surface, atomic H in oxide and metal
# Reuses solve() -> sweep() -> thickness_figure() above; only the solver is new.
# =============================================================================
from calculations.surface_kinetics import (get_all_properties, solve_steady_state_flux_L1L6,
                                           solve_steady_state_flux_L2aL6, solve_steady_state_flux_direct,
                                           calculate_mixed_defect_flux_L6, calculate_defective_metal_flux_L6,
                                           calculate_full_model_flux_L346_v2)
from calculations.config.model_config import METALS, get_surface_kinetics_from_config

MICRO   = SIM['microstructure']
DEFECTS = SIM['defect_config']
SIEVERTS_PINHOLE = SIM['oxide_defects'].get('use_sieverts_pinhole', False)

# Temperature is fixed, so the Model 2 properties are computed once
PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], T_op)       # oxide surface + transport
SM    = get_surface_kinetics_from_config(SIM['metal_name'], T_op, METALS)    # bare-metal surface

LEVELS_M2 = {   # level: (title, layers in the wall)
    'L1L6':  ('Level 1+6: Perfect metal + surface kinetics',                     {'metal'}),
    'L2aL6': ('Level 2a+6: Perfect oxide alone + surface kinetics',              {'oxide'}),
    'L2bL6': ('Level 2b+6: Perfect oxide + perfect metal + surface kinetics',    {'oxide', 'metal'}),
    'L3L6':  ('Level 3+6: Defective oxide + perfect metal + surface kinetics',   {'oxide', 'metal'}),
    'L4L6':  ('Level 4+6: Perfect oxide + defective metal + surface kinetics',   {'oxide', 'metal'}),
    'L5L6':  ('Level 5+6: Defective oxide + defective metal + surface kinetics', {'oxide', 'metal'}),
}
LEVELS.update(LEVELS_M2)

def solve_m2(level, Lox, Lm):
    """Flux and regime label for one Model 2 level with the given layer thicknesses."""
    k, Keq, Dox, Kox, Dm, Ksm = (PROPS[key] for key in ('k_diss', 'K_eq', 'D_ox', 'K_ox', 'D_m', 'K_s_m'))
    pin  = {} if SIEVERTS_PINHOLE else dict(k_diss_metal=SM['k_diss'], K_eq_metal=SM['K_eq'])
    mode = MICRO.get('mode', 'both')
    if level == 'L1L6':  r = solve_steady_state_flux_L1L6(P_up, P_down, Lm, SM['k_diss'], SM['K_eq'], Dm, Ksm)
    if level == 'L2aL6': r = solve_steady_state_flux_L2aL6(P_up, P_down, k, Keq, Dox, Kox, Lox)
    if level == 'L2bL6': r = solve_steady_state_flux_direct(P_up, P_down, Lm, k, Keq, Dox, Kox, Lox, Dm, Ksm)
    if level == 'L3L6':  r = calculate_mixed_defect_flux_L6(P_up, P_down, Lm, k, Keq, Dox, Kox, Lox,
                                                            Dm, Ksm, DEFECTS, **pin)
    if level == 'L4L6':  r = calculate_defective_metal_flux_L6(P_up, P_down, Lm, T_op, k, Keq, Dox, Kox, Lox,
                                                               Dm, Ksm, microstructure_params=MICRO, mode=mode)
    if level == 'L5L6':  r = calculate_full_model_flux_L346_v2(P_up, P_down, Lm, T_op, k, Keq, Dox, Kox, Lox,
                                                               Dm, Ksm, MICRO, DEFECTS, mode=mode, **pin)
    return r.get('J_total', r.get('J_ss', r.get('flux'))), label_model2(level, r, DEFECTS)

for level in LEVELS_M2:
    fig, fits = thickness_figure(level)
    save_figure(fig, f"{level}_thickness", FIGS)
    fits.to_csv(FIGS / f"{level}_thickness_fits.csv", index=False)

## 7. Three temperatures and three pressures: the thickness sweeps at the low / mid / high of each range

In [ ]:
# =============================================================================
# THREE TEMPERATURES (at P_up) and THREE PRESSURES (at T_op): low / mid / high of each range
# Same figure as above, with one curve per condition in each panel. Colour and marker still
# mean the regime; line style and the label at the end of each curve mean the condition.
# =============================================================================
from contextlib import contextmanager

T_lo, T_hi = CONDITIONS['T_range']
P_lo, P_hi = CONDITIONS['P_range']
SETS = {   # set: (title note, [(curve label, at_conditions arguments), ...])
    'T': ('three temperatures (low / mid / high of T_range),   ' + rf'$P_{{up}}$ = {P_up:.0e} Pa',
          [(f'{Tc:.0f} K', dict(Tc=Tc)) for Tc in (T_lo, (T_lo + T_hi) / 2, T_hi)]),    # 573, 923, 1273 K
    'P': ('three pressures (low / mid / high of P_range),   ' + rf'$T$ = {T_op:.0f} K',
          [(p_tag(Pc), dict(Pc=Pc)) for Pc in (P_lo, np.sqrt(P_lo * P_hi), P_hi)]),    # 1e-7, 1, 1e7 Pa
}

@contextmanager
def at_conditions(Tc=None, Pc=None):
    """Run solve() and sweep() above at another (T, P_up); the study values are restored afterwards."""
    global T_op, P_up, OX, ME, PROPS, SM
    saved = T_op, P_up, OX, ME, PROPS, SM
    T_op  = T_op if Tc is None else Tc
    P_up  = P_up if Pc is None else Pc
    OX    = get_oxide_properties_at_T(SIM['oxide_name'], T_op)
    ME    = get_metal_properties_at_T(SIM['metal_name'], T_op)
    PROPS = get_all_properties(SIM['oxide_name'], SIM['metal_name'], T_op)
    SM    = get_surface_kinetics_from_config(SIM['metal_name'], T_op, METALS)
    try:
        yield
    finally:
        T_op, P_up, OX, ME, PROPS, SM = saved

for level in LEVELS:
    for kind in SETS:
        note, conds = SETS[kind]
        curves = {row: [] for row in ROWS}
        for tag, kw in conds:
            with at_conditions(**kw):
                for row in ROWS:
                    curves[row].append((tag, sweep(level, row)))
        fig, fits = thickness_figure(level, curves, note)
        save_figure(fig, f"{level}_thickness_3{kind}", FIGS)
        fits.to_csv(FIGS / f"{level}_thickness_3{kind}_fits.csv", index=False)